# 01 · Evaluación Traditional RAG
Evalúa el Traditional RAG (ChromaDB + MiniLM + Gemini) sobre UltraDomain con métricas RAGAS.

In [1]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../..'))
os.listdir()

['04_eval_RAG++.ipynb',
 '02_eval_LightRAG.ipynb',
 '01_eval_RAG.ipynb',
 '05_eval_msGraphRAG.ipynb',
 '00_DatasetLoading.ipynb',
 '03_eval_PropertyRAG.ipynb']

## 1. Configuración del experimento

In [2]:
DOMINIO     = "cs"   # dominio de UltraDomain
N_LIBROS    = None      # cuántos libros indexar
MAX_Q       = None      # preguntas a evaluar (None = todas)
SHUFFLE     = False  # True = libros aleatorios
CARPETA_DB  = "../../chroma_db_eval_traditional"
RESULTS_DIR = "../../results_traditionalRAG/"

## 2. Cargar datos

In [3]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   Total Q&A: 100


In [4]:
# En vez de cargar el primer libro, cargar el que corresponde a las Q&A
# DOMINIO  = "cs"
# N_LIBROS = 1
# SHUFFLE  = False

# libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

# Verificar que coinciden
print(f"Libro indexado: {libros[0]['titulo']}")
print(f"Q&A son sobre: {qas[0]['titulo']}")

Libro indexado: Introducing Regular Expressions
Q&A son sobre: Machine Learning With Spark


## 3. Inicializar y indexar el RAG

In [5]:
import shutil
from src.data_loader import load_and_split_text
from src.baselines.traditional_rag import TraditionalRAG

rag = TraditionalRAG(persist_directory=CARPETA_DB)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cargando índice existente desde ../../chroma_db_eval_traditional...


In [6]:
for libro in libros:
    # Guardar texto en fichero temporal para load_and_split_text
    tmp = f"/tmp/{libro['context_id']}.txt"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(libro["texto"])
    splits = load_and_split_text(tmp)
    print(f"📖 {libro['titulo']} → {len(splits)} fragmentos")
    rag.index_documents(splits)

print("✅ Indexación completada")

📖 Introduction to the Theory of Programming Languages → 266 fragmentos
Creando nuevo índice de vectores...
📖 Introducing Regular Expressions → 320 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Linux Kernel Networking → 1849 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Machine Learning With Spark → 764 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Joe Celko's SQL Programming Style → 425 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Professional Microsoft SQL Server 2008 Programming → 2461 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Mastering VBA for Microsoft Office 2013 → 2747 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Probability and Statistics for Computer Science → 1658 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Guide to Java → 845 fragmentos
Añadiendo nuevos documentos al índice existente...
📖 Modern Optimization With R → 489 fragmentos
Añadiendo nu

## 4. Ejecutar evaluación

In [ ]:
from src.evaluation.experiment import run_experiment 

result = await run_experiment(
    rag_type="traditional",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📚 Dominio: cs
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   Total Q&A: 100

🔍 Evaluando TRADITIONAL | cs | 100 preguntas
────────────────────────────────────────────────────────────
  [1/100] How does Spark Streaming enable real-time data processing?...
  [2/100] What does the book suggest about the use of histograms in data analysi..

## 5. Inspeccionar respuestas individuales

In [ ]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")

❓ What is the significance of the R tool in the context of modern optimization methods?
✅ GT:  The R tool is significant because it is a free, open-source, and multi-platform tool specifically developed for statistical analysis, and it has an ac...
🤖 RAG: The R tool is significant in the context of modern optimization methods for several reasons:

1.  **Practical Implementation:** It serves as a practic...
⏱️  4.95s

❓ What are the three main approaches to handle multi-objective tasks discussed in the book?
✅ GT:  The three main approaches discussed are the weighted-formula approach, lexicographic approach, and Pareto front approach....
🤖 RAG: The three main approaches to handle multi-objective tasks discussed in the book are: weighted-formula, lexicographic, and Pareto front....
⏱️  1.43s

❓ Can you name some popular modern optimization methods discussed in the book?
✅ GT:  Some popular modern optimization methods discussed in the book include simulated annealing, tabu search, genetic